# 15 · Workflows – agents that follow a plan

One agent with many tools gets confused. A **workflow** splits the job into steps, each with a focused agent, and defines **who goes next**. This chapter uses **Microsoft Agent Framework**, Foundry's recommended way to build workflows (the portal's visual designer is retired after 2026-12-01).

The key design question is **who picks the next step**:

| Pattern | Who picks the next step | Use for | Section |
|---|---|---|---|
| **Sequential** (A → B → C) | **your code**: fixed edges | pipelines, document processing | 1 · invoice pipeline, with a threshold branch |
| **Concurrent** (fan out, merge) | **your code**: everyone at once, then an aggregator | briefings, gather and summarise | 2 · morning briefing |
| **Group chat** | a **selector** function (or an orchestrator LLM) | review boards, debate | 3 · review board |
| **Handoff** | the **agents**, only along the hand-offs you allow | triage, specialists | 4 · workshop support |
| **Declarative YAML** | your code: the same fixed flow, as a file | reviewable, versioned flows | 5 · invoice pipeline as YAML |
| Magentic | a manager agent | open-ended tasks | not covered |

In every pattern a model decides what to *say* and which *tools* to call **inside** a step. With fixed edges it never decides which step comes next.

Sections 1–5 run **in your Python process**, calling Foundry models. Section 6 hosts a workflow **in Foundry** as a hosted agent. Deep dive: [docs/workflows.md](../docs/workflows.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *
import stark_docs

fix_dotnet_for_powerfx()   # only changes anything for an x64 Python on Windows-on-ARM (needed in section 5)

from typing import Annotated
from pydantic import Field
from agent_framework import Agent, AgentResponseUpdate, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder, GroupChatBuilder, GroupChatState, HandoffBuilder

client = FoundryChatClient(project_endpoint=cfg("STARK_PROJECT_ENDPOINT"), model=model("chat"), credential=credential())

## Tools for the pipeline
Plain Python functions. `approval_mode="never_require"` lets the agent call them without a human (fine for read-only tools). The logic itself comes from [stark_docs.py](stark_docs.py), the same functions as in chapter 13.

In [ ]:
@tool(approval_mode="never_require")
def extract_invoice_fields(path: Annotated[str, Field(description="Invoice file, e.g. data/docs/invoice-pym-4471.pdf")]) -> str:
    """Read an invoice with Document Intelligence and return fields, line items and confidences as JSON."""
    import json
    return json.dumps(extract_invoice(path), default=str)

@tool(approval_mode="never_require")
def check_approval_limit(amount: Annotated[float, Field(description="Invoice total in USD")],
                         approver: Annotated[str, Field(description="e.g. 'pepper potts'")]) -> str:
    """Check whether an amount is within the approver's limit."""
    import json
    return json.dumps(stark_docs.check_approval_limit(amount, approver))

invoice_review = skill_body("invoice-review")   # the SKILL.md checklist without its front matter (chapter 07)

## 1 · Sequential: the invoice pipeline
Three specialists. The reviewer gets the **invoice-review skill** (chapter 07) injected as instructions. `intermediate_output_from` makes the first two steps visible, not only the last.

> ℹ️ `SequentialBuilder` is the shortcut: the list order is the run order. Below, the **same chain is wired by hand** with `WorkflowBuilder`; the order is fixed either way.

In [ ]:
extractor = client.as_agent(name="extractor", tools=[extract_invoice_fields],
    instructions="Extract the invoice. Call extract_invoice_fields with the path from the user, then list vendor, number, date, total and any low-confidence (<0.8) field.")
reviewer = client.as_agent(name="reviewer", tools=[check_approval_limit],
    instructions="You review the extracted invoice. Follow this checklist exactly:\n" + invoice_review +
                 "\nUse check_approval_limit with approver 'pepper potts'.")
reporter = client.as_agent(name="reporter",
    instructions="Write a three-bullet message for Pepper Potts: what was reviewed, the decision, what she must do next.")

def build_pipeline():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return SequentialBuilder(participants=[extractor, reviewer, reporter], intermediate_output_from=[extractor, reviewer]).build()

async def show(workflow, task):
    last = None
    async for event in workflow.run(task, stream=True):
        if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
            u = event.data
            if u.author_name != last:
                print(f"\n\n[{'FINAL' if event.type == 'output' else 'step'}] {u.author_name}: ", end="")
                last = u.author_name
            print(u.text, end="", flush=True)
    print()

await show(build_pipeline(), "Process the invoice at data/docs/invoice-pym-4471.pdf")

Run it again with `data/docs/invoice-oscorp-9930.pdf` (about 99,792 USD): the same flow now ends in a different decision, because the checklist and the limit tool decide, not luck.

> ⚠️ Build a **new workflow for every run** (`build_pipeline()`). A workflow object keeps its conversation, and running the same object a second time fails with `No tool call found for function call output`. The agents can be reused.

In [ ]:
await show(build_pipeline(), "Process the invoice at data/docs/invoice-oscorp-9930.pdf")

### The same chain, wired by hand
`SequentialBuilder(participants=[a, b, c])` is a shortcut for a **linear graph**. With `WorkflowBuilder` you write the graph yourself: wrap each agent in an `AgentExecutor` (a box) and join the boxes with `add_edge(from, to)` (an arrow). The order is hard-wired in code, and the model never changes it.

| | `SequentialBuilder` | `WorkflowBuilder` + `add_edge` |
|---|---|---|
| order | list order | every edge written out |
| what each agent sees | the conversation so far | your choice per step: `AgentExecutor(agent, context_mode=...)` with `full`, `last_agent` or `custom` |
| beyond a straight line | no | conditions on edges, fan-out, custom code steps |
| used in | section 1, the first runs | section 1 (this cell), and the **hosted** version in section 6 |

`output_from` marks the box whose result is the workflow's answer, and `intermediate_output_from` shows progress from the earlier boxes.

In [ ]:
from agent_framework import AgentExecutor, WorkflowBuilder

def build_pipeline_with_edges():
    # the same three agents as above, now as boxes joined by arrows
    step1, step2, step3 = (AgentExecutor(a) for a in (extractor, reviewer, reporter))
    return (WorkflowBuilder(name="invoice_pipeline", start_executor=step1, output_from=[step3],
                            intermediate_output_from=[step1, step2])
            .add_edge(step1, step2)   # extractor -> reviewer
            .add_edge(step2, step3)   # reviewer  -> reporter
            .build())

await show(build_pipeline_with_edges(), "Process the invoice at data/docs/invoice-pym-4471.pdf")

Same agents, same answer, same fixed order. Use the shortcut for a plain chain. Reach for the explicit version when you need to control what each step sees, add a condition or a code step, or add a final step that returns plain text for hosting (section 6).

### Branch on a threshold: your code decides
The old Foundry workflow designer had condition nodes ("if the output says X, go to Y"). In Agent Framework the same thing is an **edge with a condition**: a plain Python function reads the previous step's output and picks the next step. No model is asked which branch to take.

The extractor answers in a fixed JSON shape (`response_format`), so the code can read the total as a number. Then the invoice goes one of three ways:

| Total (USD) | Next step | Who acts |
|---|---|---|
| up to 5,000, and no low-confidence field | `auto_approve` | code only, no model |
| above 50,000 | `escalate` | code only, hands it to Tony |
| anything else | `reviewer` → `reporter` | the agents from above |

| Piece | What it does here |
|---|---|
| `default_options={"response_format": InvoiceFacts}` | the extractor must answer as JSON that matches the Pydantic model `InvoiceFacts` |
| `add_switch_case_edge_group(extract, [Case(...), Case(...), Default(...)])` | checks the cases in order and sends the message to the **first** one whose condition is true; `Default` takes the rest |
| `@executor` | a step that is plain Python: it reads the facts and returns a decision without calling a model |
| `output_from=[auto_approve, escalate, report]` | every branch ends in a step whose result is the workflow's answer |

The thresholds are function arguments, so you can change them per run and watch the path change.

In [ ]:
from typing import Never
from pydantic import BaseModel
from agent_framework import AgentExecutorResponse, Case, Default, WorkflowContext, executor

class InvoiceFacts(BaseModel):
    vendor: str
    invoice_number: str
    total_usd: float
    low_confidence_fields: list[str]

facts_extractor = client.as_agent(name="facts_extractor", tools=[extract_invoice_fields],
    default_options={"response_format": InvoiceFacts},
    instructions="Call extract_invoice_fields with the path from the user. Return vendor, invoice number, total in USD "
                 "and the names of the fields with a confidence below 0.8.")

def facts(response: AgentExecutorResponse) -> InvoiceFacts:
    """The extractor's answer as typed data. The routing reads these fields, never free text."""
    return InvoiceFacts.model_validate_json(response.agent_response.text)

@executor(id="auto_approve")
async def auto_approve(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """A code step: no model is called."""
    f = facts(response)
    await ctx.yield_output(f"APPROVE (automatic): {f.vendor} {f.invoice_number}, {f.total_usd:,.2f} USD. No model reviewed it.")

@executor(id="escalate")
async def escalate(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """A code step: no model is called."""
    f = facts(response)
    await ctx.yield_output(f"ESCALATE to Tony Stark: {f.vendor} {f.invoice_number}, {f.total_usd:,.2f} USD is above the escalation threshold.")

def build_router(auto_up_to=5_000, escalate_above=50_000):
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    extract, review, report = AgentExecutor(facts_extractor), AgentExecutor(reviewer), AgentExecutor(reporter)
    return (WorkflowBuilder(name="invoice_router", start_executor=extract, output_from=[auto_approve, escalate, report])
            .add_switch_case_edge_group(extract, [
                # checked in order, the first true condition wins
                Case(condition=lambda r: facts(r).total_usd <= auto_up_to and not facts(r).low_confidence_fields, target=auto_approve),
                Case(condition=lambda r: facts(r).total_usd > escalate_above, target=escalate),
                Default(target=review),   # everything in between gets the agent review
            ])
            .add_edge(review, report)
            .build())

async def route(task, **thresholds):
    """Run the router and print the answer and the path it took."""
    path = []
    async for event in build_router(**thresholds).run(task, stream=True):
        if event.type == "executor_invoked" and event.executor_id not in path:
            path.append(event.executor_id)
        elif event.type == "output":
            print(event.data if isinstance(event.data, str) else event.data.text, end="", flush=True)
    print("\npath:", " -> ".join(path))

In [ ]:
await route("Process the invoice at data/docs/invoice-pym-4471.pdf")                       # about 3,332 USD
print()
await route("Process the invoice at data/docs/invoice-oscorp-9930.pdf")                    # about 99,792 USD
print()
await route("Process the invoice at data/docs/invoice-pym-4471.pdf", auto_up_to=1_000)     # same invoice, lower threshold

Read the `path:` lines:

- **Pym** (about 3,332 USD) goes `facts_extractor -> auto_approve`: one model call to read the invoice, then code decides. If Document Intelligence reports a low-confidence field, it goes to the reviewer instead, because the rule says so.
- **Oscorp** (about 99,792 USD) goes `facts_extractor -> escalate`, and the reviewer and reporter never run.
- **Pym with `auto_up_to=1_000`** goes `facts_extractor -> reviewer -> reporter`. The invoice is the same, only the threshold changed.

The model still reads the invoice, but it does not choose the route. Your code does, so you can unit-test the rule without a model.

> ⚠️ If the extractor returns JSON that does not match `InvoiceFacts`, `facts()` raises and the run fails. That is the safe choice for money: an invoice the code cannot read is never auto-approved. To route it to a person instead, catch the error in a condition and send it to a review step.

## 2 · Concurrent: J.A.R.V.I.S.'s morning briefing
In a **concurrent** workflow every participant gets the **same input at the same time**, and an **aggregator** merges their answers. Three specialists check the workshop in parallel (suits, workshop systems, pending invoices); J.A.R.V.I.S. turns their reports into one briefing for Tony.

| Piece | What it does here |
|---|---|
| `suits`, `workshop`, `finance` | each calls the **stark-mcp** tools (chapter 06) through the project connection `stark-mcp`, with only the tools it needs |
| `ConcurrentBuilder(participants=[...])` | fans the request out to all three at once |
| `.with_aggregator(brief)` | runs when all three are done; the J.A.R.V.I.S. agent writes the merged briefing, and its text is the workflow's output |

`client.get_mcp_tool(..., project_connection_id="stark-mcp")` is a **hosted** MCP tool: Foundry calls the server and uses the connection's function key, so the key never reaches your code. The `store: False` option keeps the agents stateless, so the same code can be hosted in section 6.

In [ ]:
NO_STORE = {"store": False}

def stark(allowed):
    """The stark-mcp tools through the project connection (chapter 06). Each specialist only gets the tools it needs."""
    return client.get_mcp_tool(name="stark", url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                               approval_mode="never_require", allowed_tools=allowed)

def build_briefing():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    suits = client.as_agent(name="suits", default_options=NO_STORE, tools=[stark(["list_suits", "get_suit"])], instructions=
        "You are the suit officer. Use your tools: list the suits, then check each one. Report which suits are ready for duty in two sentences.")
    workshop = client.as_agent(name="workshop", default_options=NO_STORE, tools=[stark(["get_workshop_status"])], instructions=
        "You are the workshop officer. Check the arc reactor, the fabrication bay and the hangar with your tool. Report in two sentences.")
    finance = client.as_agent(name="finance", default_options=NO_STORE, tools=[stark(["check_invoice_approval"])], instructions=
        "You are the finance officer. Two invoices wait for Pepper Potts: Pym Particle Supply 3,332.34 USD and Oscorp Materials 99,792.00 USD. "
        "Check both with your tool. Report in two sentences.")
    jarvis = client.as_agent(name="jarvis", default_options=NO_STORE, instructions=
        "You are J.A.R.V.I.S., dry and witty, addressing Tony Stark as sir. Merge the three reports into a morning briefing of at most five lines. "
        "Mention anything that needs his attention.")

    async def brief(results):
        reports = chr(10).join(f"[{r.executor_id}] {r.agent_response.text}" for r in results)
        return (await jarvis.run(reports)).text

    return (ConcurrentBuilder(name="morning_briefing", participants=[suits, workshop, finance],
                              intermediate_output_from=[suits, workshop, finance])
            .with_aggregator(brief).build())

In [ ]:
import time

async def brief_me(task):
    reports, final, started = {}, [], time.time()
    async for event in build_briefing().run(task, stream=True):
        data = event.data
        if event.type == "intermediate" and isinstance(data, AgentResponseUpdate) and data.text:
            reports[data.author_name] = reports.get(data.author_name, "") + data.text
        elif event.type == "output":
            final.append(data if isinstance(data, str) else data.text)
    for name, text in reports.items():
        print(f"[{name}] {text.strip()}")
        print()
    print("[J.A.R.V.I.S.]", " ".join(final).strip())
    print(f"({time.time() - started:.0f}s)")

await brief_me("Good morning, J.A.R.V.I.S. Brief me.")

The three reports are collected **in whatever order they finish**, and the total time is about that of the slowest specialist plus the merge, not the sum of all four. The specialists never see each other's answers: only J.A.R.V.I.S. does.

> ℹ️ The data behind `stark-mcp` is static on purpose, so the briefing is the same every morning, apart from J.A.R.V.I.S.'s wording.

## 3 · Group chat: the review board
Agents share one conversation; a **selector** picks the next speaker (here round-robin; use `orchestrator_agent=` for an LLM that decides). A termination condition ends the chat.

In [ ]:
engineer = client.as_agent(name="engineer", instructions="You are a suit engineer. Judge technical need in two sentences.")
accountant = client.as_agent(name="accountant", instructions="You are the accountant. Judge cost and budget in two sentences.")
security = client.as_agent(name="security", instructions="You are the security officer. Judge supply-chain and safety risk in two sentences. If all three of you agree, end with 'DECISION: <approve|reject>'.")

def round_robin(state: GroupChatState) -> str:
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]

board = GroupChatBuilder(
    participants=[engineer, accountant, security],
    termination_condition=lambda conversation: sum(1 for m in conversation if m.role == "assistant") >= 6,
    intermediate_output_from=[engineer, accountant, security],
    selection_func=round_robin,
).build()

await show(board, "Should we buy 4 vials of Pym particles for 3,332 USD for the shrink-ray prototype?")

## 4 · Handoff: triage and specialists
In a **handoff**, agents decide to pass control based on the conversation; the user stays in the loop. `triage` routes each message to a specialist, the specialist answers, and control can go back to `triage`.

> ⚠️ Every participant must be created with `require_per_service_call_history_persistence=True`. Without it `build()` raises `Handoff workflows require all participant agents to have ...`. The flag keeps each agent's local history consistent with the service when control is handed over.

`enable_return_to_previous(False)` makes `triage` look at **every** user message. Without it, the next message goes straight to whoever spoke last.

In [ ]:
# handoff workflows need every participant to keep its history per service call
triage = client.as_agent(name="triage", require_per_service_call_history_persistence=True, instructions="Route the user: suits and hardware -> suit_engineer; access, badges, threats -> security_officer. Otherwise answer yourself.")
suit_engineer = client.as_agent(name="suit_engineer", require_per_service_call_history_persistence=True, instructions="You fix suits. Be practical. Hand back to triage when done.")
security_officer = client.as_agent(name="security_officer", require_per_service_call_history_persistence=True, instructions="You handle security incidents. Be strict. Hand back to triage when done.")

def build_handoff():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return (
        HandoffBuilder(name="workshop_support", participants=[triage, suit_engineer, security_officer])
        .with_start_agent(triage)
        .enable_return_to_previous(False)
        .add_handoff(triage, [suit_engineer, security_officer])
        .add_handoff(suit_engineer, [triage])
        .add_handoff(security_officer, [triage])
        .build()
    )

print("handoff workflow built:", build_handoff().name)

### Run it with scripted user turns
A handoff workflow **pauses and asks the user** after each specialist answers: you get a `request_info` event with a `HandoffAgentUserRequest`. You answer it with `HandoffAgentUserRequest.create_response(text)`, or end the chat with `HandoffAgentUserRequest.terminate()`. The helper below plays a list of user messages instead of calling `input()`, and prints who speaks, so you can see the handoffs.

In [ ]:
from agent_framework.orchestrations import HandoffAgentUserRequest

async def chat(workflow, user_turns):
    """Run a handoff workflow with scripted user turns; print who speaks and where control goes."""
    turns = list(user_turns)
    print("USER:", turns[0])
    responses = None
    first = turns.pop(0)
    while True:
        pending, last = [], None
        stream = workflow.run(first, stream=True) if responses is None else workflow.run(responses=responses, stream=True)
        async for event in stream:
            if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
                pending.append(event)
            elif event.type == "output" and isinstance(event.data, AgentResponseUpdate) and event.data.text:
                if event.data.author_name != last:
                    print(f"\n[{event.data.author_name}] ", end="")
                    last = event.data.author_name
                print(event.data.text, end="", flush=True)
        print()
        if not pending:
            break
        if turns:
            nxt = turns.pop(0)
            print("\nUSER:", nxt)
            responses = {r.request_id: HandoffAgentUserRequest.create_response(nxt) for r in pending}
        else:
            responses = {r.request_id: HandoffAgentUserRequest.terminate() for r in pending}
        first = None

In [ ]:
await chat(build_handoff(), [
    "My Mark 42 repulsor is jammed and will not fire.",
    "Thanks. Now someone used a stolen badge to enter the vault yesterday.",
    "Thanks, that is all.",
])

Read the speaker labels: a hardware question goes to `suit_engineer`, a badge question to `security_officer`, and the closing thanks is answered by `triage`. An agent may sometimes answer a question itself, so the exact route can vary between runs. For a real chat, replace the list with `input(...)`.

## 5 · The same pipeline as YAML
Declarative workflows keep the *flow* in a reviewable file; agents are registered by name. (Needs `agent-framework-declarative`, Python 3.10–3.13 and a **.NET runtime**: expressions like `=System.ConversationId` are evaluated by PowerFx. If you see `PowerFx is not available (dotnet runtime not installed)` although `dotnet` is installed, your Python and your .NET have different CPU architectures; `fix_dotnet_for_powerfx()` in the first cell handles the common x64-Python-on-ARM case. Restart the kernel after changing it.)

In [ ]:
print((ROOT / "workflows" / "invoice-pipeline.yaml").read_text())

In [ ]:
from agent_framework.declarative import WorkflowFactory
factory = WorkflowFactory()
factory.register_agent("ExtractorAgent", extractor)
factory.register_agent("ReviewerAgent", reviewer)
factory.register_agent("ReporterAgent", reporter)
declared = factory.create_workflow_from_yaml_path(ROOT / "workflows" / "invoice-pipeline.yaml")
result = await declared.run("Process the invoice at data/docs/invoice-pym-4471.pdf")
for out in result.get_outputs():
    print(out)

## 6 · Run it in Foundry
Foundry's own workflow designer is being retired (2026-12-01), and its YAML is a different dialect from the file in section 5. The supported route is to host your **Agent Framework** workflow as a **hosted agent**, like chapter 09: your code runs in a managed container, with an endpoint, an identity, versions and traces.

### Two finished projects

| Hosted agent | Pattern | What it needs |
|---|---|---|
| [invoice-workflow](../src/invoice-workflow/main.py) | the sequential chain of section 1, wired with `WorkflowBuilder` | Document Intelligence, so the agent's identity needs a role (see *Deploy*) |
| [morning-briefing](../src/morning-briefing/main.py) | the concurrent briefing of section 2 | the `stark-mcp` project connection from chapter 06, nothing else |

### What changes when you host a workflow

| In the hosted project | Why |
|---|---|
| `ResponsesHostServer(workflow=build_workflow, parse_response=...)` | hosts the workflow **natively**. `workflow.as_agent()` is deprecated, because a `WorkflowAgent` keeps state and must not serve different users |
| `build_workflow(request)` builds a **fresh** workflow per request | a workflow keeps its conversation (the same rule as `build_pipeline()` in section 1) |
| `default_options={"store": False}` on the agents | the host keeps the history; native workflows cannot checkpoint provider-side storage |
| the output is **plain text**: a final executor, or the aggregator's return value | raw streamed model updates cannot be a native workflow output (the host rejects them with `cannot checkpoint unfinished provider background output`) |

### Deploy

From a terminal in the repo root:

```bash
azd deploy invoice-workflow          # zips src/invoice-workflow, Foundry builds it, creates version 1
azd deploy morning-briefing          # same steps for the concurrent briefing
python scripts/grant_agent_role.py   # once, for invoice-workflow only: let its identity call Document Intelligence
```

> ⚠️ A deployed agent runs as **its own identity**, not yours. `invoice-workflow` fails with `PermissionDenied` until [grant_agent_role.py](../scripts/grant_agent_role.py) gives that identity the *Cognitive Services User* role (allow a minute or two to take effect). `morning-briefing` needs nothing: its MCP calls go through the project connection.

### Call them

A hosted agent is called like any agent, with a client **bound** to it (chapter 04). The bound client has its own conversations, so the helper creates and deletes the conversation with the same client. Each call takes **30–40 seconds**: several model and tool calls per request.

In [ ]:
def ask_hosted(agent_name, text):
    bound = openai_client(agent_name=agent_name)
    conversation = bound.conversations.create()
    try:
        return bound.responses.create(conversation=conversation.id, input=text).output_text
    finally:
        bound.conversations.delete(conversation.id)

print(ask_hosted("invoice-workflow", "Process the invoice at data/docs/invoice-pym-4471.pdf"))

In [ ]:
print(ask_hosted("morning-briefing", "Good morning, J.A.R.V.I.S. Brief me."))

> ⏱️ The first call after a container has been idle can be slower. If it answers `proxy_timeout` (HTTP 424, "agent container did not respond within the timeout window"), just call again.

## 👀 In the portal

**Build → Agents → invoice-workflow** and **morning-briefing**: versions, the playground, and **Traces** with one trace per call. `azd deploy` prints a direct link. They appear as hosted agents; Not checked: whether the portal draws the workflow graph (the local Agent Inspector in VS Code does).

## 🧹 Clean up
The hosted agents stay deployed. Uncomment the lines to remove them, or run `azd down --purge` for everything.

In [ ]:
# delete_agent("invoice-workflow")
# delete_agent("morning-briefing")

## 🎯 Recap

| Need | Pattern | Who picks the next step |
|---|---|---|
| Fixed steps | sequential | your code |
| Branch on the data | conditional edges (`Case` / `Default`) | your code |
| Fan out, then merge | concurrent + aggregator | your code |
| Debate / several viewpoints | group chat | a selector |
| Route to the right specialist, user in the loop | handoff | the agents |
| Reviewable flow in git | declarative YAML | your code |
| Run in Foundry | native hosting (`workflow=`) as a hosted agent | the same as the pattern you host |

> 💡 Next: [16 · Vision – agents calling agents](16_vision_a2a_capstone.ipynb) 🧪.